# Lab 17 — Image Compression with SVD

## How a picture becomes a few important matrix layers

This lab is a full computational companion to Chapter 17. We will treat images as matrices, decompose them into SVD layers, compress them, measure error, visualize singular values, denoise images, and end with a high-dimensional view of low-rank structure.

The main idea is:

> A matrix image can often be approximated by a small number of meaningful rank-one layers.

## 0. Setup

We use only standard scientific Python libraries. The lab creates synthetic images so that it can run anywhere without downloading external data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(42)

## 1. A grayscale image is a matrix

We first build a synthetic image. It contains a smooth background, a bright disk, a diagonal shadow, and a stripe pattern. This is simple enough to understand but rich enough to show compression behavior.

In [ ]:
def make_synthetic_image(n=160):
    x = np.linspace(-1, 1, n)
    y = np.linspace(-1, 1, n)
    X, Y = np.meshgrid(x, y)

    background = 0.25 + 0.25*(X + 1)/2 + 0.15*(Y + 1)/2
    disk = 0.55 * ((X + 0.28)**2 + (Y - 0.15)**2 < 0.33**2)
    ring = 0.35 * (((X - 0.35)**2 + (Y + 0.20)**2 < 0.28**2) & ((X - 0.35)**2 + (Y + 0.20)**2 > 0.18**2))
    stripes = 0.12*np.sin(10*np.pi*X + 4*np.pi*Y)
    stripes *= (np.abs(Y + 0.45) < 0.12)
    shadow = -0.22 * (Y > X + 0.35)

    A = background + disk + ring + stripes + shadow
    return np.clip(A, 0, 1)

A = make_synthetic_image(160)

plt.figure(figsize=(5,5))
plt.imshow(A, cmap='gray', vmin=0, vmax=1)
plt.title('Synthetic grayscale image')
plt.axis('off')
plt.show()

print('Shape:', A.shape)
print('Number of pixels:', A.size)
print('Minimum and maximum brightness:', A.min(), A.max())

### Student reflection

1. Which parts of this image look smooth?
2. Which parts look like edges?
3. Which parts look repetitive?
4. Which parts do you expect to be easy or hard to compress?

## 2. Compute the SVD

The SVD writes the image matrix as

$$
A = U\Sigma V^T.
$$

Equivalently,

$$
A = \sigma_1u_1v_1^T + \sigma_2u_2v_2^T + \cdots.
$$

The singular values tell us the importance of each layer.

In [ ]:
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print('U shape:', U.shape)
print('number of singular values:', len(s))
print('Vt shape:', Vt.shape)
print('First 10 singular values:')
print(s[:10])

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(s, marker='o', markersize=3)
plt.title('Singular values of the image')
plt.xlabel('Index')
plt.ylabel('Singular value')
plt.grid(True)
plt.show()

plt.figure(figsize=(7,4))
plt.semilogy(s, marker='o', markersize=3)
plt.title('Singular values on a log scale')
plt.xlabel('Index')
plt.ylabel('Singular value')
plt.grid(True)
plt.show()

### Interpretation

A fast drop in singular values means that the first few layers capture a large part of the matrix energy. A slow drop means information is spread across many layers.

## 3. Rank-one layers

The first SVD layer is

$$
\sigma_1 u_1v_1^T.
$$

Each layer is a rank-one image. It may not look like a natural image by itself, but it is one ingredient in the reconstruction.

In [ ]:
def layer(i):
    return s[i] * np.outer(U[:, i], Vt[i, :])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(8)):
    Li = layer(i)
    vmax = np.max(np.abs(Li))
    ax.imshow(Li, cmap='gray', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Layer {i+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

Notice that early layers usually contain broad structure. Later layers often contain more localized detail, texture, or noise-like corrections.

## 4. Rank-$k$ approximations

Now we build

$$
A_k = \sum_{i=1}^k \sigma_i u_i v_i^T.
$$

As $k$ increases, the reconstructed image becomes closer to the original.

In [ ]:
def rank_k_approx(U, s, Vt, k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

ks = [1, 2, 5, 10, 20, 40, 80, 120]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, k in zip(axes.ravel(), ks):
    Ak = rank_k_approx(U, s, Vt, k)
    ax.imshow(np.clip(Ak, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Rank {k}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Student task

Change the list `ks`. Try values such as `3`, `8`, `15`, `30`, and `60`. Which value of $k$ gives a good visual tradeoff between quality and storage?

## 5. Storage ratio and energy captured

A full $m	imes n$ image stores $mn$ numbers.

A rank-$k$ SVD approximation stores approximately

$$
k(m+n+1)
$$

numbers.

The energy captured by the first $k$ singular values is

$$
rac{\sigma_1^2+\cdots+\sigma_k^2}{\sigma_1^2+\cdots+\sigma_r^2}.
$$

In [ ]:
m, n = A.shape
full_storage = m*n

def storage_ratio(k):
    return k*(m+n+1)/full_storage

def energy_ratio(k):
    return np.sum(s[:k]**2)/np.sum(s**2)

print(f'Full storage: {full_storage} numbers')
print(' k | storage ratio | energy captured')
print('---|---------------|----------------')
for k in [1, 2, 5, 10, 20, 40, 80, 120]:
    print(f'{k:3d}| {storage_ratio(k):13.3f} | {energy_ratio(k):14.3f}')

In [ ]:
ks_all = np.arange(1, len(s)+1)
energy_all = np.cumsum(s**2)/np.sum(s**2)
storage_all = np.array([storage_ratio(k) for k in ks_all])

plt.figure(figsize=(7,4))
plt.plot(ks_all, energy_all, label='energy captured')
plt.plot(ks_all, storage_all, label='storage ratio')
plt.axhline(1, linestyle='--')
plt.title('Energy captured versus storage ratio')
plt.xlabel('rank k')
plt.legend()
plt.grid(True)
plt.show()

### Interpretation

Good compression happens when energy rises quickly while storage grows slowly.

## 6. Reconstruction error

The Frobenius error satisfies

$$
\|A-A_k\|_F^2 = \sum_{i=k+1}^r \sigma_i^2.
$$

We can verify this numerically.

In [ ]:
print(' k | direct error | singular-value error')
print('---|--------------|---------------------')
for k in [1, 5, 10, 20, 40, 80]:
    Ak = rank_k_approx(U, s, Vt, k)
    direct_error = np.linalg.norm(A - Ak, 'fro')
    sv_error = np.sqrt(np.sum(s[k:]**2))
    print(f'{k:3d}| {direct_error:12.6f} | {sv_error:19.6f}')

## 7. Comparing easy and hard images

Some images are easy to compress; some are hard. Let us compare a smooth image with a random image.

In [ ]:
def smooth_image(n=160):
    x = np.linspace(-1, 1, n)
    y = np.linspace(-1, 1, n)
    X, Y = np.meshgrid(x, y)
    return np.clip(0.45 + 0.25*X + 0.15*np.cos(2*np.pi*Y) + 0.2*np.exp(-6*(X**2+Y**2)), 0, 1)

S = smooth_image(160)
R = rng.random((160,160))

images = {'structured image': A, 'smooth image': S, 'random image': R}

plt.figure(figsize=(12,4))
for i, (name, M) in enumerate(images.items(), 1):
    plt.subplot(1,3,i)
    plt.imshow(M, cmap='gray', vmin=0, vmax=1)
    plt.title(name)
    plt.axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
for name, M in images.items():
    _, sv, _ = np.linalg.svd(M, full_matrices=False)
    plt.plot(np.cumsum(sv**2)/np.sum(sv**2), label=name)
plt.title('Energy captured curves')
plt.xlabel('rank k')
plt.ylabel('energy captured')
plt.legend()
plt.grid(True)
plt.show()

### Student question

Why does the random image need many more singular values? What does that tell us about structure?

## 8. Denoising with truncated SVD

Now we add noise to the image and try to remove it by keeping only the strongest SVD layers.

In [ ]:
noise_level = 0.18
A_noisy = np.clip(A + noise_level*rng.standard_normal(A.shape), 0, 1)

Un, sn, Vtn = np.linalg.svd(A_noisy, full_matrices=False)

ks = [5, 10, 20, 40, 80]
plt.figure(figsize=(14, 6))
plt.subplot(2,3,1)
plt.imshow(A_noisy, cmap='gray', vmin=0, vmax=1)
plt.title('Noisy image')
plt.axis('off')

for idx, k in enumerate(ks, 2):
    Dk = rank_k_approx(Un, sn, Vtn, k)
    plt.subplot(2,3,idx)
    plt.imshow(np.clip(Dk, 0, 1), cmap='gray', vmin=0, vmax=1)
    plt.title(f'Denoised rank {k}')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print(' k | error to clean image')
print('---|---------------------')
for k in [5, 10, 20, 40, 80, 120]:
    Dk = np.clip(rank_k_approx(Un, sn, Vtn, k), 0, 1)
    err = np.linalg.norm(A - Dk, 'fro')
    print(f'{k:3d}| {err:19.4f}')

### Interpretation

For denoising, the best $k$ is not always the largest $k$. If $k$ is too large, we begin to reconstruct the noise as well as the signal.

## 9. Color image compression using three channels

A color image is an array with shape `(height, width, 3)`. We create a synthetic color image and compress each channel separately.

In [ ]:
def make_color_image(n=160):
    x = np.linspace(-1, 1, n)
    y = np.linspace(-1, 1, n)
    X, Y = np.meshgrid(x, y)
    R = np.clip(0.5 + 0.35*X + 0.25*((X+0.35)**2 + (Y-0.15)**2 < 0.25**2), 0, 1)
    G = np.clip(0.45 + 0.3*Y + 0.25*np.sin(3*np.pi*X)*np.cos(2*np.pi*Y), 0, 1)
    B = np.clip(0.6 - 0.25*X + 0.20*((X-0.25)**2 + (Y+0.2)**2 < 0.35**2), 0, 1)
    return np.dstack([R, G, B])

C = make_color_image(160)
plt.figure(figsize=(5,5))
plt.imshow(C)
plt.title('Synthetic color image')
plt.axis('off')
plt.show()

In [ ]:
def compress_channel(M, k):
    Uc, sc, Vtc = np.linalg.svd(M, full_matrices=False)
    return rank_k_approx(Uc, sc, Vtc, k)

def compress_color_image(C, k):
    channels = [compress_channel(C[:,:,j], k) for j in range(3)]
    return np.clip(np.dstack(channels), 0, 1)

ks = [1, 3, 8, 20, 50]
plt.figure(figsize=(14, 6))
for i, k in enumerate(ks, 1):
    plt.subplot(1, len(ks), i)
    plt.imshow(compress_color_image(C, k))
    plt.title(f'k = {k}')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 10. A high-dimensional view: low-rank signal plus noise

Image compression is one example of a broader idea: data may live near a low-dimensional structure inside a high-dimensional space.

We create a high-dimensional matrix whose true signal has rank $3$, then add noise.

In [ ]:
m, n = 120, 80
true_rank = 3
L = rng.normal(size=(m, true_rank))
R = rng.normal(size=(true_rank, n))
Signal = L @ R
Noise = 0.5*rng.normal(size=(m, n))
Data = Signal + Noise

Ud, sd, Vtd = np.linalg.svd(Data, full_matrices=False)

plt.figure(figsize=(7,4))
plt.plot(sd, marker='o')
plt.title('Singular values of low-rank signal plus noise')
plt.xlabel('index')
plt.ylabel('singular value')
plt.grid(True)
plt.show()

print('First 10 singular values:')
print(sd[:10])

In [ ]:
for k in [1,2,3,5,10,20]:
    Dk = rank_k_approx(Ud, sd, Vtd, k)
    err_to_signal = np.linalg.norm(Signal - Dk, 'fro') / np.linalg.norm(Signal, 'fro')
    print(f'k={k:2d}: relative error to true signal = {err_to_signal:.4f}')

### Final reflection

1. Why does $k=3$ have special meaning in this synthetic example?
2. How is this connected to denoising?
3. How is this connected to PCA?
4. Why is SVD useful beyond images?

## 11. Mini-project

Choose one of the following.

### Option A: compression report

Use the synthetic image `A`. Compare $k=5,10,20,40,80$. For each $k$, report:

- storage ratio,
- energy captured,
- Frobenius error,
- visual quality.

Write a short paragraph recommending a value of $k$.

### Option B: denoising report

Use the noisy image `A_noisy`. Compare several $k$ values. Find a value that removes noise while preserving structure. Explain your choice.

### Option C: color compression

Use `C`. Compress it channel-by-channel. Explain how quality changes as $k$ increases.